In [3]:
# !pip -q install sentence-transformers chromadb

In [4]:
# -- IMPORTS --
from pathlib import Path
import json
import chromadb
from sentence_transformers import SentenceTransformer

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
base = Path("/content/drive/MyDrive/geant_chatbot")
docs_dir = base / "processed" / "docs_json"
vector_dir = base / "vectordb"

print("Docs:", len(list(docs_dir.glob("*.json"))))
print("Vector DB path:", vector_dir)

Docs: 56
Vector DB path: /content/drive/MyDrive/geant_chatbot/vectordb


In [6]:
def chunk_text(text, chunk_size=1200, overlap=200):
    chunks = []
    start = 0
    text_len = len(text)

    while start < text_len:
        end = min(start + chunk_size, text_len)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += (chunk_size - overlap)

    return chunks

In [7]:
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(EMBED_MODEL)

client = chromadb.PersistentClient(path=str(vector_dir))
collection = client.get_or_create_collection(name="geant_corpus")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
chunk_count = 0
doc_count = 0

for doc_file in docs_dir.glob("*.json"):
    doc = json.loads(doc_file.read_text(encoding="utf-8"))

    doc_id = doc["doc_id"]
    text = doc["text"]
    source_path = doc["source_path"]
    doc_type = doc["doc_type"]

    chunks = chunk_text(text)

    ids = []
    documents = []
    metadatas = []

    for i, chunk in enumerate(chunks):
        ids.append(f"{doc_id}::chunk{i}")
        documents.append(chunk)
        metadatas.append({
            "doc_id": doc_id,
            "chunk_index": i,
            "source_path": source_path,
            "doc_type": doc_type})

    embeddings = model.encode(documents, show_progress_bar=False).tolist()

    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=embeddings)

    chunk_count += len(chunks)
    doc_count += 1

    print(f"Indexed {doc_file.name}: {len(chunks)} chunks")

print("\nDone.")
print("Documents indexed:", doc_count)
print("Total chunks:", chunk_count)


Indexed txt__connect_article_8.json: 5 chunks
Indexed txt__connect_article_5.json: 5 chunks
Indexed txt__connect_article_49.json: 3 chunks
Indexed pdf__GN5-2 Intelligent Networks The Rise of Generative AI in Network Management_GN5-2-White-Paper_Gen-AI-in-Network-Management.json: 149 chunks
Indexed pdf__Switch OAV Architecture Analysis_GN5-2_White-Paper_Switch-OAV-Architecture-Analysis.json: 27 chunks
Indexed pdf__GN5-2 MARnet OAV Architecture Analysis_GN5-2-White-Paper_MARnet-OAV-Architecture-Analysis.json: 32 chunks
Indexed pdf__GN5-2 WFO Telemetry Module Incubator Project Closure Report_GN5-1_WFO-Telemetry-Module-Incubator-Project-Closure-Report_Public.json: 10 chunks
Indexed pdf__GN5-2 Fibre Sensing Technologies Users and Use Cases for NRENs_GN5-2-White-Paper_Fibre-Sensing-Technologies.json: 57 chunks
Indexed pdf__GN5-2 D21 Project Communications Strategy and Plan_GN5-2_D2-1_Project-Communications-Strategy-and-Plan.json: 42 chunks
Indexed pdf__GN5-2 D32 Compendium Report_GN5-2_D3.2_

In [9]:
query = "What is the GN5-2 Project Communications Strategy about?"

q_emb = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=q_emb,
    n_results=5,
    include=["documents", "metadatas"])

for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0]), 1):
    print(f"\n--- Result {i} ---")
    print("Source:", meta["source_path"])
    print(doc[:400])



--- Result 1 ---
Source: /content/drive/MyDrive/geant_chatbot/raw_docs/zenodo_docs/GN5-2 D21 Project Communications Strategy and Plan_GN5-2_D2-1_Project-Communications-Strategy-and-Plan.pdf
t ID: GN5-2-25-C7321A Contents Executive Summary 1 1 Introduction 2 2 Project Communications Strategy 3 3 Project Communications Plan 5 3.1 Strategic Considerations 5 3.1.1 Audiences 5 3.1.2 Channels 6 3.1.3 Formats 6 3.1.4 Design 9 3.1.5 Messaging 9 3.1.6 Stakeholder Engagement 10 3.2 Communications Plan 12 4 Key Performance Indicators 18 5 Conclusions 19 Glossary 20 References 21 Figures Figure 

--- Result 2 ---
Source: /content/drive/MyDrive/geant_chatbot/raw_docs/zenodo_docs/GN5-2 D21 Project Communications Strategy and Plan_GN5-2_D2-1_Project-Communications-Strategy-and-Plan.pdf
(WP2 T1) to progress and enhance the work it started in GN5-1 (Section 2). The communications plan compiled by the Task is based on the communications strategy, and key communication aspects as well as stakeholder imp